# Quant Risk Core: Financial Mathematics & Visualizations
This notebook demonstrates the core mathematical frameworks implemented in the `quant_risk_core` repository.

## 1. Volatility Clustering: GARCH(1,1)
The GARCH(1,1) model captures time-varying volatility:
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$
where $\omega > 0, \alpha \ge 0, \beta \ge 0$ and $\alpha + \beta < 1$ for covariance stationarity.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from market_risk.volatility import GARCHEngine

np.random.seed(42)
returns = np.random.normal(0, 0.01, 1000)
returns[200:250] *= 3
returns[700:750] *= 2
returns_series = pd.Series(returns)

garch = GARCHEngine(p=1, q=1)
garch.fit(returns_series)
vol = garch.conditional_volatility()

fig = go.Figure()
fig.add_trace(go.Scatter(y=returns, name='Log Returns', opacity=0.5))
fig.add_trace(go.Scatter(y=vol, name='GARCH Volatility', line=dict(color='red')))
fig.update_layout(title='GARCH(1,1) Conditional Volatility', xaxis_title='Days', yaxis_title='Magnitude')
fig.show()

## 2. Value at Risk (VaR) & Expected Shortfall (ES)
VaR represents the quantile of the loss distribution:
$$\text{VaR}_\alpha = \inf \{ L : P(L > L) \le 1 - \alpha \}$$
ES represents the average loss exceeding VaR:
$$\text{ES}_\alpha = E[L | L > \text{VaR}_\alpha]$$

In [2]:
from market_risk.estimators import RiskEngine

engine = RiskEngine(confidence_levels=[0.95, 0.99])
hist_risk = engine.historical_var_es(returns_series)

fig = go.Figure()
fig.add_trace(go.Histogram(x=returns, nbinsx=50, name='Returns Distribution'))
fig.add_vline(x=-hist_risk['VaR_0.95'], line_dash='dash', line_color='orange', annotation_text='VaR 95%')
fig.add_vline(x=-hist_risk['VaR_0.99'], line_dash='dash', line_color='red', annotation_text='VaR 99%')
fig.update_layout(title='Returns Distribution with VaR Thresholds', xaxis_title='Log Return', yaxis_title='Frequency')
fig.show()

## 3. Extreme Value Theory (EVT)
Using the Peak-Over-Threshold (POT) method, we model tail excesses using the Generalized Pareto Distribution (GPD):
$$G_{\xi, \beta}(x) = 1 - (1 + \xi x / \beta)^{-1/\xi}$$
This allows for better tail estimation than normal distributions.

In [3]:
from market_risk.extreme_value import EVTEngine

evt = EVTEngine(threshold_quantile=0.95)
evt.fit(returns_series)
var_evt, es_evt = evt.estimate_risk(alpha=0.99)

print(f'EVT 99% VaR: {var_evt:.4f}')
print(f'EVT 99% ES : {es_evt:.4f}')

EVT 99% VaR: 0.0317
EVT 99% ES : 0.0418


## 4. Credit Value Adjustment (CVA)
CVA aggregates counterparty risk over a forward timeline:
$$\text{CVA} \approx (1 - R) \sum_{i=1}^n \text{EE}(t_i) \Delta \text{PD}(t_{i-1}, t_i)$$
where $\text{EE}$ is Expected Exposure and $\text{PD}$ is Probability of Default.

In [4]:
from credit_risk.counterparty import CounterpartyRiskEngine

time_grid = np.linspace(0, 5, 50)
np.random.seed(42)
paths = np.cumsum(np.random.normal(0, 1, (1000, 50)), axis=1) # Brownian Motion paths
cpty = CounterpartyRiskEngine(time_grid)
cpty.set_portfolio_paths(paths)
profiles = cpty.calculate_exposure_profiles(quantile=0.99)

fig = go.Figure()
fig.add_trace(go.Scatter(x=time_grid, y=profiles['EE'], name='Expected Exposure (EE)'))
fig.add_trace(go.Scatter(x=time_grid, y=profiles['PFE'], name='Potential Future Exposure (PFE) 99%', line=dict(dash='dash')))
fig.update_layout(title='Counterparty Exposure Profiles', xaxis_title='Years', yaxis_title='Exposure Value')
fig.show()